# Random Embedding Baseline

This notebook computes a minimal lower-bound baseline for the AnlamVer semantic similarity experiment. Each unique word receives a fixed-seed random 768-dimensional vector, pair similarities are computed with cosine similarity, and the resulting similarities are compared with human `Sim` scores using Spearman correlation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

PROJECT_ROOT = Path.cwd()
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "anlamver_tokenization_analysis.csv"
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "anlamver-final.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PAIR_OUTPUT_PATH = RESULTS_DIR / "0901-random_baseline_pair_similarities.csv"
SUMMARY_OUTPUT_PATH = RESULTS_DIR / "0902-random_baseline_summary.csv"

BASELINE_NAME = "random_baseline"
EMBEDDING_DIM = 768
SEED = 42

## Load AnlamVer Pairs

The processed tokenization-analysis file is preferred because it already contains the cleaned AnlamVer columns used elsewhere in the project. If it is unavailable, the raw AnlamVer CSV is loaded with the same semicolon delimiter, Turkish Windows encoding, and comma-decimal handling used in the main experiment notebooks.

In [ ]:
def clean_anlamver_pairs(frame: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"W1", "W2", "Sim"}
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    pairs = frame[["W1", "W2", "Sim"]].copy()
    pairs["W1"] = pairs["W1"].astype("string").str.strip()
    pairs["W2"] = pairs["W2"].astype("string").str.strip()

    if not pd.api.types.is_numeric_dtype(pairs["Sim"]):
        pairs["Sim"] = pairs["Sim"].astype("string").str.replace(",", ".", regex=False)
    pairs["Sim"] = pd.to_numeric(pairs["Sim"], errors="coerce")

    pairs = pairs.dropna(subset=["W1", "W2", "Sim"]).reset_index(drop=True)
    pairs = pairs[(pairs["W1"] != "") & (pairs["W2"] != "")].reset_index(drop=True)
    return pairs


if PROCESSED_PATH.exists():
    pairs = clean_anlamver_pairs(pd.read_csv(PROCESSED_PATH))
    data_source = PROCESSED_PATH
else:
    pairs = clean_anlamver_pairs(pd.read_csv(RAW_PATH, encoding="cp1254", sep=";", decimal=","))
    data_source = RAW_PATH

print(f"Loaded {len(pairs)} AnlamVer pairs from {data_source}")
pairs.head()

## Assign Random Word Vectors

In [ ]:
unique_words = sorted(pd.unique(pd.concat([pairs["W1"], pairs["W2"]], ignore_index=True)))

rng = np.random.default_rng(SEED)
embedding_matrix = rng.normal(size=(len(unique_words), EMBEDDING_DIM)).astype(np.float32)
embedding_norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
normalized_embeddings = embedding_matrix / embedding_norms
word_to_index = {word: index for index, word in enumerate(unique_words)}

print(f"Created random embeddings for {len(unique_words)} unique words")

## Compute Pair Similarities and Spearman Correlation

In [ ]:
w1_indices = pairs["W1"].map(word_to_index).to_numpy()
w2_indices = pairs["W2"].map(word_to_index).to_numpy()
random_cosine_similarities = np.sum(
    normalized_embeddings[w1_indices] * normalized_embeddings[w2_indices],
    axis=1,
)

pair_results = pairs.copy()
pair_results["baseline_name"] = BASELINE_NAME
pair_results["embedding_dim"] = EMBEDDING_DIM
pair_results["seed"] = SEED
pair_results["random_cosine_similarity"] = random_cosine_similarities

spearman_rho, p_value = spearmanr(pair_results["Sim"], pair_results["random_cosine_similarity"])

summary = pd.DataFrame(
    [
        {
            "baseline_name": BASELINE_NAME,
            "embedding_dim": EMBEDDING_DIM,
            "seed": SEED,
            "n_words": len(unique_words),
            "n_pairs": len(pair_results),
            "spearman_rho": spearman_rho,
            "p_value": p_value,
        }
    ]
)

pair_results.to_csv(PAIR_OUTPUT_PATH, index=False, encoding="utf-8")
summary.to_csv(SUMMARY_OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Saved pair-level random baseline results to: {PAIR_OUTPUT_PATH}")
print(f"Saved random baseline summary to: {SUMMARY_OUTPUT_PATH}")

In [ ]:
print(f"Random embedding baseline Spearman rho: {spearman_rho:.6f}")
print(f"Random embedding baseline p-value: {p_value:.6g}")

## Interpretation

“The random embedding baseline provides a chance-level lower-bound comparison. Because the vectors contain no lexical or semantic information, a Spearman correlation close to zero indicates that random vector similarities do not systematically align with human similarity judgments.”